In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import random
import shutil
from ultralytics import YOLO
from PIL import Image
import yaml
import matplotlib.pyplot as plt
import cv2
from IPython.display import Image, display

In [ ]:
images_dir = 'dataset/images'  # image folder
labels_dir = 'dataset/labels'  # label folder
output_base = 'dataset/pokemon dataset'
split_ratio = 0.8  # Train/val split

image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png'))]
random.shuffle(image_files)

split_index = int(len(image_files) * split_ratio)
train_files = image_files[:split_index]
val_files = image_files[split_index:]

def copy_files(file_list, subset):
    img_out_dir = os.path.join(output_base, 'images', subset)
    lbl_out_dir = os.path.join(output_base, 'labels', subset)
    os.makedirs(img_out_dir, exist_ok=True)
    os.makedirs(lbl_out_dir, exist_ok=True)

    for fname in file_list:
        name, _ = os.path.splitext(fname)
        label_file = name + '.txt'
        
        shutil.copy(os.path.join(images_dir, fname), os.path.join(img_out_dir, fname))
        shutil.copy(os.path.join(labels_dir, label_file), os.path.join(lbl_out_dir, label_file))

copy_files(train_files, 'train')
copy_files(val_files, 'val')

print(f"Split complete: {len(train_files)} train, {len(val_files)} val")

In [ ]:
base_path = r'dataset'
dataset_path = os.path.join(base_path, 'pokemon dataset')
class_file = os.path.join(base_path, 'classes.txt')
yaml_path = os.path.join(base_path, 'data.yaml')

with open(class_file, 'r') as f:
    class_names = [line.strip() for line in f if line.strip()]

# Build YAML dictionary
data_yaml = {
    'train': os.path.join(dataset_path, 'images/train').replace('\\', '/'),
    'val': os.path.join(dataset_path, 'images/val').replace('\\', '/'),
    'nc': len(class_names),
    'names': class_names
}

with open(yaml_path, 'w') as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=False)

print(f"data.yaml created at: {yaml_path}")

In [ ]:
model = YOLO('yolov8n.pt')

# Train the model
model.train(
    data='dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16
)

In [ ]:
display(Image(filename="train results/results.png"))
display(Image(filename="train results/labels.jpg"))
display(Image(filename="train results/F1_curve.png"))
display(Image(filename="train results/P_curve.png"))
display(Image(filename="train results/R_curve.png"))
display(Image(filename="train results/PR_curve.png"))

In [ ]:
# Testing 1

model = YOLO("train results/weights/best.pt")

input_path = "testvid.mp4"     
output_path = "results.mp4"

# Run prediction and save the result
model.predict(
    source=input_path,
    save=True,          
    save_txt=False,     
    conf=0.25,          
)

In [ ]:
# Testing 2

model = YOLO("train results/weights/best.pt")

input_path = "test.mp4"     
output_path = "results.mp4"

# Run prediction and save the result
model.predict(
    source=input_path,
    save=True,          
    save_txt=False,     
    conf=0.25,          
)

In [ ]:
# Testing on an image

model = YOLO("train results/weights/best.pt")

results = model.predict(
    source="testimg.jpg", 
    save=True,
    conf=0.25,
    project="runs",  
    name="results_image" 
)

In [ ]:
display(Image(filename="testimgresult.jpg"))